# Singular Value Decomposition (SVD): $A = Q \Sigma P^T$

## 1. Overview and Problem Definition

Given the rectangular $2 \times 3$ matrix $A$:
$$A = \begin{bmatrix} 2 & 0 & 1 \\ -1 & 2 & 9 \end{bmatrix}$$

We want to calculate its **Singular Value Decomposition (SVD)**:
$$A = Q \Sigma P^T$$

where:
- **$Q \in \mathbb{R}^{2 \times 2}$**: Orthogonal matrix ($Q^T Q = Q Q^T = I_2$) whose columns are the **left singular vectors** (eigenvectors of $A A^T$).
- **$\Sigma \in \mathbb{R}^{2 \times 3}$**: Rectangular diagonal matrix containing the **singular values** $\sigma_1 \ge \sigma_2 \ge 0$ on the main diagonal.
- **$P \in \mathbb{R}^{3 \times 3}$**: Orthogonal matrix ($P^T P = P P^T = I_3$) whose columns are the **right singular vectors** (eigenvectors of $A^T A$).
- **$P^T \in \mathbb{R}^{3 \times 3}$**: Transpose of $P$.

---

### The Tableau Form (Lecture 04, Slide 15)
$$[N \times M] = [N \times N] \; [N \times M] \; [M \times M]$$
$$[2 \times 3] = [2 \times 2] \; [2 \times 3] \; [3 \times 3]$$

- **$[2 \times 2] \implies A A^T$ provides $Q$**
- **$[3 \times 3] \implies A^T A$ provides $P$ and $P^T$**

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([
    [ 2.0, 0.0, 1.0],
    [-1.0, 2.0, 9.0]
])

print("Matrix A [2x3]:")
print(A)

Matrix A [2x3]:
[[ 2.  0.  1.]
 [-1.  2.  9.]]


## 2. Step 1: Compute Gram Matrices $A A^T$ and $A^T A$

### Why compute both $A A^T$ and $A^T A$?
Substituting the SVD definition $A = Q \Sigma P^T$ into the two products reveals why:
$$A A^T = (Q \Sigma P^T)(Q \Sigma P^T)^T = (Q \Sigma P^T)(P \Sigma^T Q^T)$$
Because $P$ is orthogonal ($P^T P = I_3$), $P$ collapses:
$$A A^T = Q (\Sigma \Sigma^T) Q^T \quad \in \mathbb{R}^{2 \times 2}$$
**Conclusion 1:** The columns of $Q$ are the orthonormal eigenvectors of the $[2 \times 2]$ matrix $A A^T$!

Similarly, multiplying in reverse order:
$$A^T A = (P \Sigma^T Q^T)(Q \Sigma P^T)$$
Because $Q$ is orthogonal ($Q^T Q = I_2$), $Q$ collapses:
$$A^T A = P (\Sigma^T \Sigma) P^T \quad \in \mathbb{R}^{3 \times 3}$$
**Conclusion 2:** The columns of $P$ are the orthonormal eigenvectors of the $[3 \times 3]$ matrix $A^T A$!

Furthermore:
$$\Sigma \Sigma^T = \begin{bmatrix} \sigma_1^2 & 0 \\ 0 & \sigma_2^2 \end{bmatrix}, \qquad \Sigma^T \Sigma = \begin{bmatrix} \sigma_1^2 & 0 & 0 \\ 0 & \sigma_2^2 & 0 \\ 0 & 0 & 0 \end{bmatrix}$$
The non-zero eigenvalues of both $A A^T$ and $A^T A$ are identical: $\lambda_i = \sigma_i^2$.

In [2]:
A_AT = A @ A.T
AT_A = A.T @ A

print("A @ A.T [2x2] (used for Q):")
print(A_AT)
print("\nA.T @ A [3x3] (used for P):")
print(AT_A)

A @ A.T [2x2] (used for Q):
[[ 5.  7.]
 [ 7. 86.]]

A.T @ A [3x3] (used for P):
[[ 5. -2. -7.]
 [-2.  4. 18.]
 [-7. 18. 82.]]


## 3. Step 2: Compute Singular Values $\sigma_i = \sqrt{\lambda_i}$

We find the eigenvalues of $A A^T$ (or $A^T A$) and sort them in descending order:
$$\lambda_1 \ge \lambda_2 > 0$$
The singular values are:
$$\sigma_1 = \sqrt{\lambda_1}, \qquad \sigma_2 = \sqrt{\lambda_2}$$

In [3]:
# Eigendecomposition of A A^T (2x2)
eigvals_Q, eigvecs_Q = np.linalg.eig(A_AT)

# Sort descending
sort_idx_Q = np.argsort(eigvals_Q)[::-1]
eigvals_Q_sorted = eigvals_Q[sort_idx_Q]

print("Sorted eigenvalues of A @ A.T:")
print(eigvals_Q_sorted)

# Singular values sigma = sqrt(eigenvalues)
singular_values = np.sqrt(eigvals_Q_sorted)
print(f"\nSingular values:")
print(f"sigma_1 = {singular_values[0]:.4f}")
print(f"sigma_2 = {singular_values[1]:.4f}")

Sorted eigenvalues of A @ A.T:
[86.6005  4.3995]

Singular values:
sigma_1 = 9.3059
sigma_2 = 2.0975


## 4. Step 3: Construct the Rectangular Diagonal Matrix $\Sigma$ ($2 \times 3$)

### Why `sigma * np.eye(2,2)` threw an error:
1. $A$ has shape $[2 \times 3]$, so $\Sigma$ must also have shape $[2 \times 3]$ to satisfy $[2 \times 3] = [2 \times 2] [2 \times 3] [3 \times 3]$.
2. Placing $\sigma_1, \sigma_2$ on the main diagonal of a $2 \times 3$ matrix with an extra zero column:
   $$\Sigma = \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \end{bmatrix}$$

In [4]:
# Form [2x3] Sigma matrix
Sigma = np.zeros_like(A, dtype=float)
Sigma[0, 0] = singular_values[0]
Sigma[1, 1] = singular_values[1]

print("Sigma matrix [2x3]:")
print(Sigma)

Sigma matrix [2x3]:
[[9.3059 0.     0.    ]
 [0.     2.0975 0.    ]]


## 5. Step 4: Compute $Q$ (from $A A^T$) and $P, P^T$ (from $A^T A$)

### The Intended Lecture Method:
1. **$Q$ from $A A^T$:** Compute the eigenvectors of $A A^T$ and arrange them according to the sorted eigenvalues $\lambda_1 \ge \lambda_2$.
2. **$P$ from $A^T A$:** Compute the eigenvectors of $A^T A$ and arrange them according to the sorted eigenvalues $\lambda_1 \ge \lambda_2 \ge \lambda_3$.
3. **Sign Alignment Check:**
   Every eigenvector has an arbitrary sign choice (both $+\mathbf{v}$ and $-\mathbf{v}$ satisfy $M \mathbf{v} = \lambda \mathbf{v}$).
   However, SVD requires:
   $$A \mathbf{p}_i = \sigma_i \mathbf{q}_i$$
   If an eigensolver chose opposite signs such that $A \mathbf{p}_i = -\sigma_i \mathbf{q}_i$, we simply flip the sign of that column in $Q$ (or $P$)!

In [5]:
# 1. Compute Q directly from A @ A.T (2x2)
Q = eigvecs_Q[:, sort_idx_Q]

# 2. Compute P directly from A.T @ A (3x3)
eigvals_P, eigvecs_P = np.linalg.eig(AT_A)
sort_idx_P = np.argsort(eigvals_P)[::-1]
P = eigvecs_P[:, sort_idx_P]

# 3. Check and align signs so that A @ p_i == sigma_i * q_i
for i in range(len(singular_values)):
    # Dot product between (A @ p_i) and q_i should be positive (+sigma_i)
    if np.dot(A @ P[:, i], Q[:, i]) < 0:
        Q[:, i] = -Q[:, i]  # flip sign to match

PT = P.T

print("Q [2x2] (from A @ A.T, columns are left singular vectors):")
print(Q)
print("\nP [3x3] (from A.T @ A, columns are right singular vectors):")
print(P)
print("\nP^T [3x3]:")
print(PT)

Q [2x2] (from A @ A.T, columns are left singular vectors):
[[-0.0855 -0.9963]
 [-0.9963  0.0855]]

P [3x3] (from A.T @ A, columns are right singular vectors):
[[ 0.0887 -0.9908 -0.1025]
 [-0.2141  0.0815 -0.9734]
 [-0.9728 -0.1083  0.2049]]

P^T [3x3]:
[[ 0.0887 -0.2141 -0.9728]
 [-0.9908  0.0815 -0.1083]
 [-0.1025 -0.9734  0.2049]]


### Shortcut Alternative for $Q$: Directly from $P$ via $\mathbf{q}_i = \frac{1}{\sigma_i} A \mathbf{p}_i$

Instead of computing a second set of eigenvectors from $A A^T$ and doing manual sign checks, you can compute $Q$ directly once you have $P$ and $\sigma_i$:
$$\mathbf{q}_i = \frac{1}{\sigma_i} A \mathbf{p}_i \quad (\text{for } \sigma_i > 0)$$

**Why does this shortcut work?**
$$(A A^T) \mathbf{q}_i = (A A^T) \left( \frac{A \mathbf{p}_i}{\sigma_i} \right) = \frac{A (A^T A \mathbf{p}_i)}{\sigma_i} = \frac{A (\sigma_i^2 \mathbf{p}_i)}{\sigma_i} = \sigma_i^2 \left( \frac{A \mathbf{p}_i}{\sigma_i} \right) = \sigma_i^2 \mathbf{q}_i$$

* It is **guaranteed** to be an eigenvector of $A A^T$.
* It is automatically **unit length** ($\|\mathbf{q}_i\| = 1$).
* Its sign is **automatically aligned** with $P$, so no sign checking is needed!

In [6]:
# Shortcut: compute q_i directly from p_i and sigma_i
q0_shortcut = (A @ P[:, 0]) / singular_values[0]
q1_shortcut = (A @ P[:, 1]) / singular_values[1]
Q_shortcut = np.column_stack([q0_shortcut, q1_shortcut])

print("Q computed via shortcut q_i = (A @ p_i) / sigma_i:")
print(Q_shortcut)
print("\nDoes shortcut Q match the lecture-method Q?", np.allclose(Q_shortcut, Q))

Q computed via shortcut q_i = (A @ p_i) / sigma_i:
[[-0.0855 -0.9963]
 [-0.9963  0.0855]]

Does shortcut Q match the lecture-method Q? True


## 6. Step 5: Verify Orthogonality of $Q$ and $P$

An orthogonal matrix satisfies:
$$Q^T Q = Q Q^T = I_2, \qquad P^T P = P P^T = I_3$$

In [7]:
print("Q.T @ Q (should be I_2):")
print(Q.T @ Q)
print("Is Q orthogonal?", np.allclose(Q.T @ Q, np.eye(2)))

print("\nP.T @ P (should be I_3):")
print(P.T @ P)
print("Is P orthogonal?", np.allclose(P.T @ P, np.eye(3)))

Q.T @ Q (should be I_2):
[[1. 0.]
 [0. 1.]]
Is Q orthogonal? True

P.T @ P (should be I_3):
[[ 1. -0. -0.]
 [-0.  1.  0.]
 [-0.  0.  1.]]
Is P orthogonal? True


## 7. Step 6: Verify Reconstruction $A = Q \Sigma P^T$

Multiplying the three factors:
$$A_{\text{reconstructed}} = Q \; \Sigma \; P^T$$

In [8]:
A_reconstructed = Q @ Sigma @ PT

print("Original Matrix A:")
print(A)

print("\nReconstructed Matrix (Q @ Sigma @ P^T):")
print(A_reconstructed)

diff = np.max(np.abs(A - A_reconstructed))
print(f"\nMax absolute error: {diff:.2e}")
print("Exact match with original A?", np.allclose(A, A_reconstructed))

Original Matrix A:
[[ 2.  0.  1.]
 [-1.  2.  9.]]

Reconstructed Matrix (Q @ Sigma @ P^T):
[[ 2. -0.  1.]
 [-1.  2.  9.]]

Max absolute error: 2.89e-15
Exact match with original A? True


## 8. Step 7: Dyadic (Outer-Product) Expansion

In Lecture 04 (Slide 29–30), SVD is expressed as a sum of rank-1 outer products:
$$A = \sum_{i=1}^{r} \sigma_i (\mathbf{q}_i \mathbf{p}_i^T) = \sigma_1 \mathbf{q}_1 \mathbf{p}_1^T + \sigma_2 \mathbf{q}_2 \mathbf{p}_2^T$$

In [9]:
comp1 = singular_values[0] * np.outer(Q[:, 0], P[:, 0])
comp2 = singular_values[1] * np.outer(Q[:, 1], P[:, 1])

print("Principal Component 1 (sigma_1 * q_1 * p_1^T):")
print(comp1)

print("\nPrincipal Component 2 (sigma_2 * q_2 * p_2^T):")
print(comp2)

print("\nSum of components (Component 1 + Component 2):")
print(comp1 + comp2)
print("Matches A?", np.allclose(comp1 + comp2, A))

Principal Component 1 (sigma_1 * q_1 * p_1^T):
[[-0.0705  0.1703  0.7737]
 [-0.8224  1.9854  9.0194]]

Principal Component 2 (sigma_2 * q_2 * p_2^T):
[[ 2.0705 -0.1703  0.2263]
 [-0.1776  0.0146 -0.0194]]

Sum of components (Component 1 + Component 2):
[[ 2. -0.  1.]
 [-1.  2.  9.]]
Matches A? True


## 9. Step 8: Comparison with NumPy Built-In `np.linalg.svd`

Compare our manually computed components with NumPy's built-in `np.linalg.svd(A)`:

In [10]:
U_np, S_np, Vt_np = np.linalg.svd(A)

print("NumPy U (corresponds to Q):")
print(U_np)

print("\nNumPy S (corresponds to singular values):")
print(S_np)

print("\nNumPy Vt (corresponds to P^T):")
print(Vt_np)

# Verify NumPy reconstruction
Sigma_np = np.zeros_like(A)
Sigma_np[0, 0] = S_np[0]
Sigma_np[1, 1] = S_np[1]
print("\nNumPy reconstruction matches A?", np.allclose(U_np @ Sigma_np @ Vt_np, A))

NumPy U (corresponds to Q):
[[ 0.0855  0.9963]
 [ 0.9963 -0.0855]]

NumPy S (corresponds to singular values):
[9.3059 2.0975]

NumPy Vt (corresponds to P^T):
[[-0.0887  0.2141  0.9728]
 [ 0.9908 -0.0815  0.1083]
 [-0.1025 -0.9734  0.2049]]

NumPy reconstruction matches A? True


## 10. Summary Table

| Matrix | Source | Role | Shape | Properties |
| :--- | :--- | :--- | :--- | :--- |
| **$A$** | Given | Original matrix | $2 \times 3$ | Rectangular |
| **$Q$** | Eigenvectors of $A A^T$ (or shortcut $\frac{A \mathbf{p}_i}{\sigma_i}$) | Left singular vectors | $2 \times 2$ | Orthogonal ($Q^{-1} = Q^T$) |
| **$\Sigma$** | $\sqrt{\text{eig}(A A^T)}$ | Singular values | $2 \times 3$ | Rectangular diagonal: $\sigma_1 \approx 9.3059$, $\sigma_2 \approx 2.0975$ |
| **$P^T$** | Eigenvectors of $A^T A$ | Right singular vectors transpose | $3 \times 3$ | Orthogonal ($P^{-1} = P^T$) |

$$A = Q \Sigma P^T = \begin{bmatrix} q_{00} & q_{01} \\ q_{10} & q_{11} \end{bmatrix} \begin{bmatrix} \sigma_1 & 0 & 0 \\ 0 & \sigma_2 & 0 \end{bmatrix} \begin{bmatrix} p_{00} & p_{10} & p_{20} \\ p_{01} & p_{11} & p_{21} \\ p_{02} & p_{12} & p_{22} \end{bmatrix}$$